# RagVision — Step 2: Web Search + Text / Image / Table Extraction

**Flow:**
1. User text or image pass → if image vision model use search-friendly query invoke.
2. LLM **router** decide web search need or not.
3. Tavily web search uses for (text + image results).
4. Top pages fetch results: **main text**, **images/charts**, **tables** extract.
5. Embedding the query image/table relevance ranking.
6. (Optional) use vision model top images verify.
7. LLM only **text answer** generate (do not image/table generate → less cost). Show the Media only or seperately .

## 1. Install & setup

In [ ]:
# !pip -q install openai tavily-python beautifulsoup4 lxml trafilatura pandas requests pillow numpy

In [2]:
! pip install trafilatura==2.3.1

In [3]:
import os, io, re, json, base64, mimetypes
from getpass import getpass
from concurrent.futures import ThreadPoolExecutor
from urllib.parse import urljoin, urlparse

import numpy as np
import pandas as pd
import requests
import trafilatura
from bs4 import BeautifulSoup
from openai import OpenAI
from tavily import TavilyClient
from IPython.display import display, HTML, Markdown

for k in ["OPENAI_API_KEY", "TAVILY_API_KEY"]:
    if not os.getenv(k):
        os.environ[k] = getpass(f"{k}: ")

oa = OpenAI()
tv = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

CHAT_MODEL = "gpt-4o-mini"
EMB_MODEL = "text-embedding-3-small"

## 2. Image query → text query, and the web-search router

In [4]:
def image_to_query(image_path: str, text: str = "") -> str:
    """User image dile seta describe kore search-friendly query banay."""
    mime = mimetypes.guess_type(image_path)[0] or "image/png"
    with open(image_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    r = oa.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role": "user", "content": [
            {"type": "text", "text": "Describe the main subject of this image in one short, search-friendly sentence "
                                     f"(name the place/object/chart if you can). User question: {text or 'none'}"},
            {"type": "image_url", "image_url": {"url": f"data:{mime};base64,{b64}", "detail": "low"}},
        ]}],
    )
    return f"{text} {r.choices[0].message.content.strip()}".strip()


def plan_search(query: str) -> dict:
    """LLM decide kore web lagbe kina + search query rewrite."""
    r = oa.chat.completions.create(
        model=CHAT_MODEL,
        response_format={"type": "json_object"},
        messages=[
            {"role": "system", "content":
                "You are a router for a RAG system. Decide whether answering needs external or fresh web info. "
                'Return JSON only: {"use_web": bool, "search_query": str, "wants_media": bool}. '
                "search_query must be short, keyword-style. wants_media is true if images/charts/tables would help."},
            {"role": "user", "content": query},
        ],
    )
    return json.loads(r.choices[0].message.content)

## 3. Web search (Tavily)

In [5]:
def web_search(query: str, k: int = 5) -> dict:
    return tv.search(
        query=query,
        max_results=k,
        search_depth= "basic", ## "advanced",
        include_images=True,
        include_image_descriptions=True,
    )

## 4. Page extractor — text, images/charts, tables

- Text: `trafilatura`
- Images: `<img>` + `og:image`; logo/icon/sprite filter, chhoto image skip, alt/caption/context save
- Charts: alt/caption/filename e `chart/graph/plot/diagram` thakle `is_chart=True`
- Tables: `pandas.read_html`

In [6]:
HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; RagVision/0.1)"}
BAD_IMG = re.compile(r"(logo|icon|sprite|avatar|pixel|tracking|badge|emoji|banner|/ads?/|\.svg|\.gif)", re.I)
CHART_HINT = re.compile(r"(chart|graph|plot|diagram|infographic|figure)", re.I)


def _img_src(tag):
    for a in ("data-src", "data-original", "data-lazy-src", "src"):
        if tag.get(a):
            return tag[a]
    ss = tag.get("srcset")
    if ss:
        return ss.split(",")[-1].strip().split(" ")[0]
    return None


def _dim(v):
    m = re.match(r"\d+", str(v or ""))
    return int(m.group()) if m else None


def extract_page(url: str, timeout: int = 12, max_imgs: int = 15, max_tables: int = 3):
    try:
        resp = requests.get(url, headers=HEADERS, timeout=timeout)
        resp.raise_for_status()
    except Exception:
        return None
    html = resp.text
    text = trafilatura.extract(html, include_comments=False, include_tables=False) or ""
    soup = BeautifulSoup(html, "lxml")
    title = soup.title.get_text(strip=True) if soup.title else url

    images, seen = [], set()

    def add_image(src, alt="", caption="", context=""):
        if not src or src.startswith("data:"):
            return
        full = urljoin(url, src)
        if full in seen or BAD_IMG.search(full):
            return
        seen.add(full)
        blob = f"{alt} {caption} {full}"
        images.append({
            "url": full, "alt": alt, "caption": caption, "context": context,
            "page_url": url, "page_title": title,
            "is_chart": bool(CHART_HINT.search(blob)),
        })

    og = soup.find("meta", property="og:image")
    if og and og.get("content"):
        add_image(og["content"], alt=title)

    for tag in soup.find_all("img"):
        w, h = _dim(tag.get("width")), _dim(tag.get("height"))
        if (w and w < 150) or (h and h < 100):
            continue
        fig = tag.find_parent("figure")
        cap = fig.find("figcaption").get_text(" ", strip=True) if fig and fig.find("figcaption") else ""
        ctx = tag.parent.get_text(" ", strip=True)[:200] if tag.parent else ""
        add_image(_img_src(tag), alt=tag.get("alt", ""), caption=cap, context=ctx)
        if len(images) >= max_imgs:
            break

    tables = []
    for t in soup.find_all("table"):
        try:
            df = pd.read_html(io.StringIO(str(t)))[0]
        except Exception:
            continue
        if df.shape[0] < 2 or df.shape[1] < 2:
            continue
        cap = t.find("caption")
        tables.append({
            "caption": cap.get_text(" ", strip=True) if cap else f"Table from {title}",
            "df": df.head(15), "page_url": url, "page_title": title,
        })
        if len(tables) >= max_tables:
            break

    return {"url": url, "title": title, "text": text, "images": images, "tables": tables}

## 5. Relevance ranking (embeddings) + optional vision verify

In [7]:
def embed(texts):
    r = oa.embeddings.create(model=EMB_MODEL, input=texts)
    return np.array([d.embedding for d in r.data])


def rank_items(query, items, text_fn, top_k):
    if not items:
        return []
    vecs = embed([query] + [(text_fn(i) or "item")[:500] for i in items])
    q, m = vecs[0], vecs[1:]
    sims = m @ q / (np.linalg.norm(m, axis=1) * np.linalg.norm(q) + 1e-9)
    for it, s in zip(items, sims):
        it["score"] = float(s)
    return sorted(items, key=lambda x: -x["score"])[:top_k]


def verify_images(query, images):
    """Optional: vision model bole image query-r sathe relevant kina (low detail = sosta)."""
    keep = []
    for im in images:
        try:
            r = oa.chat.completions.create(
                model=CHAT_MODEL,
                response_format={"type": "json_object"},
                messages=[{"role": "user", "content": [
                    {"type": "text", "text": f'Is this image relevant to: "{query}"? Return JSON {{"relevant": bool}}.'},
                    {"type": "image_url", "image_url": {"url": im["url"], "detail": "low"}},
                ]}],
            )
            if json.loads(r.choices[0].message.content).get("relevant"):
                keep.append(im)
        except Exception:
            keep.append(im)  # fetch fail hole drop na kore rakhi
    return keep

## 6. Text-only answer generation

In [8]:
def generate_answer(query, pages, tables):
    ctx = "\n\n".join(
        f"[{i+1}] {p['title']} ({p['url']})\n{p['text'][:3000]}" for i, p in enumerate(pages)
    )
    tctx = "\n\n".join(
        f"TABLE: {t['caption']}\n{t['df'].head(8).to_csv(index=False)}" for t in tables[:3]
    )
    r = oa.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content":
                "Answer using ONLY the sources below. Cite like [1], [2]. "
                "Images and tables are displayed separately by the UI, so you may say 'see the images/table below', "
                "but never try to draw or generate images, charts or tables yourself. Be concise."},
            {"role": "user", "content": f"Question: {query}\n\nSOURCES:\n{ctx}\n\n{tctx}"},
        ],
    )
    return r.choices[0].message.content

## 7. Full Step-2 pipeline

In [15]:
def web_pipeline(query, image_path=None, top_pages=4, top_images=6, top_tables=2, verify=False):
    trace = []  # frontend graph animation e ei node list use hobe
    if image_path:
        query = image_to_query(image_path, query)
        trace.append("image_to_query")

    plan = plan_search(query)
    trace.append("router")
    if not plan.get("use_web", True):
        return {"answer": None, "use_web": False, "images": [], "tables": [], "sources": [], "trace": trace}

    sq = plan.get("search_query") or query
    res = web_search(sq, k=top_pages)
    trace.append("web_search")

    urls = [r["url"] for r in res.get("results", [])]
    with ThreadPoolExecutor(max_workers=6) as ex:
        fetched = list(ex.map(extract_page, urls))
    trace.append("extract_page")

    pages = []
    for r, p in zip(res.get("results", []), fetched):
        if p is None:
            p = {"url": r["url"], "title": r.get("title", r["url"]), "text": r.get("content", ""), "images": [], "tables": []}
        elif len(p["text"]) < 200:
            p["text"] = r.get("content", "") or p["text"]
        pages.append(p)

    images = [im for p in pages for im in p["images"]]
    for im in res.get("images", []):  # Tavily-r nijer image results
        print(im)
        if isinstance(im, dict):
            images.append({"url": im.get("url",""), "alt": im.get("description", ""), "caption": "", "context": "",
                           "page_url": "", "page_title": "tavily", "is_chart": bool(CHART_HINT.search(str(im.get("description") or "")))})
        else:
            images.append({"url": im, "alt": "", "caption": "", "context": "", "page_url": "", "page_title": "tavily", "is_chart": False})
    tables = [t for p in pages for t in p["tables"]]

    images = rank_items(sq, images, lambda i: f"{i['alt']} {i['caption']} {i['context']} {i['page_title']}", top_images)
    tables = rank_items(sq, tables, lambda t: f"{t['caption']} {' '.join(map(str, t['df'].columns))}", top_tables)
    trace.append("rank_media")

    if verify and images:
        images = verify_images(sq, images)
        trace.append("verify_images")

    answer = generate_answer(query, pages, tables)
    trace.append("generate_answer")

    return {
        "answer": answer, "use_web": True, "search_query": sq,
        "images": images, "tables": tables,
        "sources": [{"n": i + 1, "title": p["title"], "url": p["url"]} for i, p in enumerate(pages)],
        "trace": trace,
    }

## 8. Notebook-e result dekhano

In [16]:
def show(result):
    if not result["use_web"]:
        display(Markdown("_Router bollo web search dorkar nei._"))
        return
    display(Markdown(result["answer"]))
    if result["images"]:
        cards = "".join(
            f"""<div style="width:230px;margin:6px;font:12px sans-serif">
                <a href="{im['url']}" target="_blank"><img src="{im['url']}" style="width:230px;height:150px;object-fit:cover;border-radius:8px"></a>
                <div>{'📊 ' if im['is_chart'] else ''}{(im['alt'] or im['caption'] or im['page_title'])[:80]}</div>
                <div style="color:#888">score {im['score']:.2f}</div></div>"""
            for im in result["images"]
        )
        display(HTML(f"<div style='display:flex;flex-wrap:wrap'>{cards}</div>"))
    for t in result["tables"]:
        display(Markdown(f"**{t['caption']}** — [{t['page_title']}]({t['page_url']})"))
        display(t["df"])
    display(Markdown("**Sources**\n" + "\n".join(f"{s['n']}. [{s['title']}]({s['url']})" for s in result["sources"])))
    print("graph trace:", " → ".join(result["trace"]))

## 9. Test

In [17]:
out = web_pipeline("Most beautiful beaches in Sardinia, Italy", verify=False)
show(out)

{'url': 'https://media.cntraveller.com/photos/6825b0da6033ac6c68f704a8/16:9/w_2560%2Cc_limit/Sardinia_150525_GettyImages-1457146882.jpg', 'title': 'Best beaches in Sardinia according to an expert | CN Traveller', 'description': 'A pristine beach with turquoise waters, surrounded by green hills and sailboats anchored in calm sea, all under a clear blue sky with mountains in the background.'}
{'url': 'https://media.cntraveller.com/photos/6825b09ebd78fd56d2199403/master/w_1600%2Cc_limit/La%2520Pelosa_Sardinia_150525_GettyImages-1459203638.jpg', 'title': 'Best beaches in Sardinia according to an expert | CN Traveller', 'description': 'A pristine beach with white sands, crystal-clear turquoise waters, a rocky coastline, and a small castle on a nearby island under a bright blue sky.'}
{'url': 'https://www.magnificentworld.com/wp-content/uploads/2023/10/Beaches-Sardinia-45-1.jpg', 'title': '10 Most Beautiful Beaches to Visit in Sardinia', 'description': None}
{'url': 'https://media.cntravelle

Some of the most beautiful beaches in Sardinia, Italy include:

1. **Chia** - Known for its windsurfing spots like Porto Campana and Su Giudeu, and stunning sea colors [1].

2. **Villasimius** - Features popular beaches like Porto Sa Ruxi and Cava Usai, renowned for their water clarity [1].

3. **Muravera (Costa Rei)** - Praised by Lonely Planet for its Caribbean-like white sands and crystal clear waters [1][2].

4. **Baunei** - Home to Cala Mariolu, with its snowflake-like sand and deep blue waters, this area is ideal for beach lovers [1].

5. **Orosei Gulf** - Offers picturesque beaches such as Oasi Bidderosa and Cala Ginepro, popular on the eastern coast [1].

6. **Calamosca Beach** - Located near Cagliari, known for its powdery white sand and family-friendly shallow waters [2].

These spots highlight the diverse beauty and appeal of Sardinia's coastline.

**Table from The 10 Most Beautiful and Best Beaches in Sardinia, Italy to Explore!** — [The 10 Most Beautiful and Best Beaches in Sardinia, Italy to Explore!](https://travel-boo.com/best-beaches-in-sardinia-italy)

,0,1,2,3
0,"Before you go, be sure to plan & book your trip!","Before you go, be sure to plan & book your trip!","Before you go, be sure to plan & book your trip!","Before you go, be sure to plan & book your trip!"
1,NaN,Browse & Book Hotels,NaN,Browse & Book Car Rentals
2,NaN,Browse & Book Flights,NaN,Browse & Book Buses and Trains


**Sources**
1. [15 Best Beaches in Sardinia Italy - Beautiful Sardinia Beaches | IB](https://italybest.com/15-best-beaches-in-sardinia)
2. [The 10 Most Beautiful and Best Beaches in Sardinia, Italy to Explore!](https://travel-boo.com/best-beaches-in-sardinia-italy)
3. [Sardinia Italy Beaches](https://www.pinterest.com/ideas/sardinia-italy-beaches/961390193817)
4. [Sardinia Italy Beaches](https://www.pinterest.ca/ideas/sardinia-italy-beaches/961390193817)

graph trace: router → web_search → extract_page → rank_media → generate_answer


In [ ]:
# Image query test (Own Docs images, it will retrive not a web search):
out = web_pipeline("explain this image?", image_path="data/p3_f0e88648b2624eb7.png")
show(out)

_Router bollo web search dorkar nei._

In [18]:
out = web_pipeline("Most beautiful beaches in Bangladesh", verify=False)
show(out)

{'url': 'https://dynamic-media-cdn.tripadvisor.com/media/photo-o/0a/a9/8b/44/sea-beach.jpg?w=500&h=500&s=1', 'title': 'THE 15 BEST Bangladesh Beaches (2026) - Tripadvisor', 'description': "A pristine beach with turquoise waters, lush green trees lining the shore, and a clear blue sky overhead, suggesting an idyllic spot among Bangladesh's most beautiful beaches."}
{'url': 'https://image.vietnamnews.vn/uploadvnnews/Article/2023/1/5/258976_cox.jpg', 'title': "World longest beach, Cox's Bazar, a must-visit place in Bangladesh", 'description': 'A stretch of pristine beach with golden sand, gentle waves, and lush greenery along the coastline, highlighting some distant buildings under a clear blue sky.'}
{'url': 'https://dynamic-media-cdn.tripadvisor.com/media/photo-o/1a/2b/4d/d1/visit-coral-island-saint.jpg?w=500&h=-1&s=1', 'title': 'Saint Martin Island West Beach - All You SHOULD Know Before Going 2026  (with Reviews)', 'description': 'A serene beach with a gently curved shoreline, tall le

Some of the most beautiful beaches in Bangladesh include:

1. **Cox’s Bazar Sea Beach** - Known as the world's longest unbroken natural sandy beach, stretching 120 km along the Bay of Bengal. It's a major tourist hub with vibrant local culture and activities [2][4].

2. **Inani Beach** - Located 27 km from Cox's Bazar, Inani is famous for its coral stones and serene waters, making it ideal for sunset views and photography [2][3].

3. **Himchari Beach** - A picturesque beach where a hillside waterfall meets the sea, providing a unique landscape for visitors [3].

4. **St. Martin's Island** - Notable for its coral beaches, coconut groves, and vibrant marine life, making it a peaceful retreat [3][4].

5. **Kuakata Beach** - Known as the 'Daughter of the Sea', Kuakata offers the rare experience of witnessing both sunrise and sunset over the sea from the same location [1][3].

6. **Patenga Beach** - Located near Chittagong, it offers a lively atmosphere with local snacks and views of the port's activity [3].

These beaches showcase a mix of natural beauty, cultural richness, and a variety of experiences suitable for all types of travelers.

**Sources**
1. [THE 15 BEST Bangladesh Beaches (2026) - Tripadvisor](https://www.tripadvisor.com/Attractions-g293935-Activities-c61-t52-Bangladesh.html)
2. [Top 15 Sea Beaches in Bangladesh You Shouldn't Miss](https://www.travelmate.com.bd/top-sea-beaches-in-bangladesh)
3. [7 Best Beaches in Bangladesh 2026 [Guide]](https://opentravelguide.com/bangladesh/beaches)
4. [12 Best Beaches in Bangladesh: Ultimate Guide for 2026](https://www.viralbangladesh.com/best-beaches-in-bangladesh)

graph trace: router → web_search → extract_page → rank_media → generate_answer


## Notes
- `verify=True` dile top images vision model diye check hoy (extra cost, kintu irrelevant image kome).
- Hotlinking block hole image dekhabe na → Step 4 frontend e backend proxy (`/img?url=`) lagbe.
- `out["trace"]` Step 4 er animated graph e use hobe.
- Step 1 er Pinecone/FAISS retrieval ei pipeline er `router` er por merge hobe (retrieve vs web decide).